In [3]:
pip install -U scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------------------------------ --------- 6.3/8.2 MB 38.6 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 39.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   -------- ------------------------------- 7.3/36.5 MB 75.3 MB/s eta 0:00:01
   -------------------- ------------------- 19.1/36.5 MB 44.7 MB/s eta 0:00:01
   ---------------------------------------  35.9/36.5 MB 57.0 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 52.7 MB/s eta 0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\DEV\dp_312\.venv\Scripts\python.exe -m pip install --upgrade pip


In [4]:
# 1. 라이브러리 불러오기 및 실습 데이터 준비
# 실습을 위해 100개의 샘플과 6개의 변수(특성)를 가진 가상 데이터를 생성.일부러 변수 간 특성을 다르게 설계하여 효과를 검증.

import pandas as pd
import numpy as np
from sklearn.datasets import make_regression

# 1. 시각화 및 데이터 핸들링용 데이터 생성
np.random.seed(42)
X_raw, y = make_regression(n_samples=100, n_features=3, noise=10, random_state=42)

df = pd.DataFrame(X_raw, columns=['Feature_A', 'Feature_B', 'Feature_C'])

# 2. [실습용 변수 추가 1] 모든 샘플의 값이 거의 일정한 변수 (분산이 매우 낮음)
df['Low_Variance_Feature'] = np.random.choice([5.0, 5.01], size=100, p=[0.95, 0.05])

# 3. [실습용 변수 추가 2] 정답(y)과 아무 상관 없는 무작위 노이즈 변수들
df['Noise_1'] = np.random.normal(0, 1, 100)
df['Noise_2'] = np.random.normal(5, 2, 100)

print("--- 원본 데이터 구조 확인 ---")
print(df.head())
print(f"원본 데이터 모양: {df.shape}") # 100행 6열

--- 원본 데이터 구조 확인 ---
   Feature_A  Feature_B  Feature_C  Low_Variance_Feature   Noise_1    Noise_2
0  -0.792521   0.504987  -0.114736                  5.00  0.087047   5.026004
1   0.280992  -0.208122  -0.622700                  5.01 -0.299007   7.907068
2   0.791032   1.402794  -0.909387                  5.00  0.091761   4.470686
3   0.625667  -1.070892  -0.857158                  5.00 -1.987569  10.440338
4  -0.342715  -0.161286  -0.802277                  5.00 -0.219672   6.251335
원본 데이터 모양: (100, 6)


In [5]:
# 2. VarianceThreshold (분산 기반 선택)
# •	방식: Filter Method (통계적 특성만 보고 판단)
# •	원리: 데이터의 값 변화가 거의 없는(분산이 0에 가까운) 변수는 모델 학습에 아무런 힌트를 주지 못하므로 사전에 제거합니다. (정답인 y 패널티는 고려하지 않고 X 자체만 검사합니다.)

from sklearn.feature_selection import VarianceThreshold

# 분산 기준치 설정 (예: 분산이 0.01 이하인 변수는 탈락)
# 모든 값이 완전히 같다면 분산은 0입니다.
selector_vt = VarianceThreshold(threshold=0.01)

# 분산 선택기 학습 및 적용
X_vt = selector_vt.fit_transform(df)

# 어떤 변수가 살아남았는지 확인
selected_support_vt = selector_vt.get_support()
features_vt = df.columns[selected_support_vt]

print("--- [VarianceThreshold] 결과 ---")
print(f"살아남은 변수 개수: {X_vt.shape[1]}개 / 원래 {df.shape[1]}개")
print(f"선택된 변수 목록: {list(features_vt)}")
# 값의 변화가 거의 없던 'Low_Variance_Feature'가 제거된 것을 확인할 수 있습니다.

--- [VarianceThreshold] 결과 ---
살아남은 변수 개수: 5개 / 원래 6개
선택된 변수 목록: ['Feature_A', 'Feature_B', 'Feature_C', 'Noise_1', 'Noise_2']


In [6]:
# 3. RFE (재귀적 특성 제거, Recursive Feature Elimination)
# 	방식: Wrapper Method (머신러닝 모델을 직접 활용)
# 	원리: 먼저 전체 변수를 넣고 모델(예: 선형 회귀, 결정 트리 등)을 학습시킨 뒤, 내부적으로 계산되는 변수 중요도(Feature Importance)나 계수(Coefficients, β)가 가장 낮은 변수를 하나씩 반복적으로 지워나갑니다. 목표로 지정한 변수 개수가 남을 때까지 수집합니다.

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# 변수 제거 과정을 함께할 기반 모델(Estimator) 객체 생성
model = LinearRegression()

# RFE 객체 생성
# n_features_to_select=3 : 최종적으로 딱 3개의 핵심 변수만 남기겠다는 의미
selector_rfe = RFE(estimator=model, n_features_to_select=3)

# RFE 학습 (여기서는 정답 데이터인 y가 반드시 필요합니다)
selector_rfe.fit(df, y)

# 결과 분석
selected_support_rfe = selector_rfe.get_support()
features_rfe = df.columns[selected_support_rfe]

print("--- [RFE] 결과 ---")
print(f"최종 선택된 3개 변수 목록: {list(features_rfe)}")
print("\n[변수별 RFE 순위 (1위가 최종 선택된 변수)]")
for col, rank in zip(df.columns, selector_rfe.ranking_):
    print(f"{col}: {rank}등")

--- [RFE] 결과 ---
최종 선택된 3개 변수 목록: ['Feature_A', 'Feature_B', 'Low_Variance_Feature']

[변수별 RFE 순위 (1위가 최종 선택된 변수)]
Feature_A: 1등
Feature_B: 1등
Feature_C: 2등
Low_Variance_Feature: 1등
Noise_1: 4등
Noise_2: 3등


In [7]:
# 4. 최종 정제된 데이터프레임 만들기
# RFE를 통해 엄선된 3개의 핵심 변수만을 추출하여 최종 머신러닝 모델용 데이터프레임을 구성합니다.

# 최종 가공 데이터 생성
df_final = df[features_rfe]

print("--- 최종 Feature Selection 완료 데이터 ---")
print(df_final.head())
print(f"최종 데이터 모양: {df_final.shape}") # 100행 3열로 압축 완료!

--- 최종 Feature Selection 완료 데이터 ---
   Feature_A  Feature_B  Low_Variance_Feature
0  -0.792521   0.504987                  5.00
1   0.280992  -0.208122                  5.01
2   0.791032   1.402794                  5.00
3   0.625667  -1.070892                  5.00
4  -0.342715  -0.161286                  5.00
최종 데이터 모양: (100, 3)
